# Post-Hoc Graphs: PASNet Pathway–Gene Network Analysis

This notebook combines outputs from three different parts of the analysis pipeline into a single integrated network view:

1. **PASNet** — the sparse pathway layer weights (`sc1`), describing how strongly each gene contributes to each Gene Ontology pathway in the trained model.
2. **SHAP feature importance** — the ranked gene list from the `PASNet_output_analysis` notebook, used here to restrict the network to the genes the model relies on most.
3. **Driver Inference** (transcription factor / signaling / non-coding driver output) and **SJARACNe** co-expression networks (TF, SIG, NC sub-networks), providing additional gene–gene relationships (mutual information, Spearman correlation) to layer on top of the PASNet pathway structure.

**What the notebook produces, step by step:**
1. Load and reshape PASNet's pathway → gene weight matrix into an edge list
2. Restrict it to the top-N SHAP-important genes and render it as a clustered heatmap
3. Load and filter the Driver Inference results table
4. Load the SJARACNe co-expression networks and merge them with the PASNet edges into one combined edge table
5. Build a node-attribute table (pathways + genes/TFs/SIGs/NCs) to attach to that edge table
6. Construct a `networkx` graph from the combined edges and node attributes
7. Filter, lay out, and draw the graph as a community-clustered network plot
8. List which nodes ended up grouped into each detected community

> **Note:** As in the previous notebook, all paths are relative to the `pipeline/output/...` directory structure produced by the Nextflow run, so this notebook is expected to sit alongside that `pipeline/` folder.


## 1. Imports

A mix of data-handling (`pandas`, `numpy`), graph (`networkx`, `igraph`, `leidenalg`), clustering (`scipy.cluster.hierarchy`, `sklearn`), and plotting (`matplotlib`, `seaborn`, `adjustText`) libraries used across the different stages below.


In [ ]:
# --- Data handling -----------------------------------------------------------
import numpy as np
import re
import pandas as pd
from sklearn.preprocessing import KBinsDiscretizer
import openpyxl

# --- Graph / network analysis -------------------------------------------------
import networkx as nx
import igraph as ig
import leidenalg

# --- Clustering ----------------------------------------------------------------
from scipy.cluster.hierarchy import linkage, leaves_list
from sklearn.preprocessing import MinMaxScaler

# --- Plotting --------------------------------------------------------------------
import matplotlib.pyplot as plt
from matplotlib.image import imread
import matplotlib.colors as mcolors
from matplotlib.colors import Normalize, LinearSegmentedColormap
import seaborn as sns
from adjustText import adjust_text

# --- General utilities -----------------------------------------------------------
import gc
import random
import math
import logging


## 2. Configuration: Genes to Highlight

`lncRNA_genes` lists gene names that should be visually highlighted (colored, bolded) wherever they appear later in the heatmap and network plots — for example to flag the long non-coding RNAs of interest among the protein-coding genes. Populate this list with the exact gene identifiers used elsewhere in the input tables.


In [ ]:
# Gene identifiers to visually flag in downstream plots (heatmap x-axis labels,
# network legend). Replace "." with the actual gene symbol(s) of interest.
lncRNA_genes = ["."]

# Number of top shap values features of interest

shap_values_features = . # eg 50


## 3. Load PASNet Pathway Weights and Reshape into an Edge List

`sc1_weights.xlsx` is the pathway-by-gene weight matrix exported from the PASNet model (see `PASNet_output_analysis.ipynb`, Section 10). It's loaded without its own header/index labels, so the row/column names are restored from `pt_fixed.xlsx`, a companion file that records which pathway and gene each row/column corresponds to.

The wide pathway × gene matrix is then **melted** into a long-format edge list (`source` = pathway, `target` = gene, `Weight` = PASNet weight), which is the standard input shape for the network/graph libraries used later. Rows with a zero weight (no learned pathway–gene connection) are dropped.


In [ ]:
# Free up memory before loading the (potentially large) weight matrix.
gc.collect()

# --- Load PASNet's sparse pathway-layer weights and their row/column labels --
sc1_go = pd.read_excel("../pipeline/output/pasnet/PASNet/Output/GO/sc1_weights.xlsx")
pt = pd.read_excel("../pipeline/output/pasnet/PASNet/Input/GO/pt_fixed.xlsx", index_col=0)

# sc1_weights.xlsx has no labels of its own; pt_fixed.xlsx supplies the
# matching pathway (row) and gene (column) names in the same order.
sc1_go.columns = pt.columns
sc1_go.index = pt.index

# --- Reshape the wide pathway x gene matrix into a long edge list ------------
data = []
for pathway, series in sc1_go.iterrows():
    for gene, value in series.items():
        data.append([pathway, gene, value])

# PASNet outputs (source: pathway; target: gene; Weight: learned PASNet weight)
df_go = pd.DataFrame(data, columns=['source', 'target', 'Weight'])

# Drop pathway-gene pairs with no learned connection (weight == 0).
df_go = df_go[df_go['Weight'] != 0]


### 3.1 Count Unique Genes Covered by the PASNet Edge List


In [ ]:
# Number of distinct genes that have at least one non-zero PASNet pathway weight.
pasnet_genes_go = df_go['target'].unique()
print(len(pasnet_genes_go))


## 4. Load Top SHAP-Ranked Genes

`feature_importance.csv` is the ranked gene table exported from the SHAP analysis in `PASNet_output_analysis.ipynb`. Here we load it and keep only the top N genes by mean absolute SHAP value, which will be used to restrict the heatmap and network to the model's most influential features.


In [ ]:
# Load the SHAP-based gene importance ranking and keep the top N genes.
shap_validation_go = pd.read_csv("../pipeline/output/pasnet/PASNet/Output/GO/feature_importance.csv", index_col=0)
shap_validation_go_top = shap_validation_go.sort_values(by="Importance", ascending=False).head(shap_values_features)

In [ ]:
# Extract the gene names (the DataFrame's index) for the top SHAP genes.
features_go = shap_validation_go_top.index

# Remove any duplicate gene names.
unique_features = pd.Index(features_go).unique()

# Store as a single-column DataFrame for convenient filtering/merging later.
features_df = pd.DataFrame(unique_features, columns=['Feature'])


## 5. Pivot the PASNet Edge List Back into a Wide Matrix

For the heatmap step, the long-format edge list is pivoted back into a wide `pathway x gene` matrix (`source` as rows, `target` as columns, `Weight` as values), with any missing pathway–gene combination filled with `0`.


In [ ]:
# Pivot the PASNet edge list into a wide pathway (rows) x gene (columns) matrix.
table_go = df_go.pivot_table(
    index='source',      # rows: pathways
    columns='target',    # columns: genes
    values='Weight',     # cell values: PASNet weight
    fill_value=0,        # missing pathway-gene pairs become 0
)

# Bring the pathway names back out of the index and into a regular column.
table_go = table_go.reset_index()


## 6. Restrict the Weight Matrix to the Top SHAP Genes

`filter_table` keeps only the gene columns that appear in the top SHAP feature list, then drops any pathway (row) left with an all-zero weight profile once those columns have been removed.


In [ ]:
def filter_table(table, features_df):
    # Use 'source' (pathway name) as the index for column-wise filtering.
    table_indexed = table.set_index('source')

    # Keep only gene columns that are also in the top SHAP feature list.
    filtered_table = table_indexed[table_indexed.columns.intersection(features_df['Feature'])]

    # Drop pathways (rows) that have no non-zero weight among the kept genes.
    filtered_table = filtered_table[filtered_table.sum(axis=1) != 0]

    # Restore 'source' as a regular column.
    return filtered_table.reset_index()

# Apply the filter to the PASNet pathway-gene weight matrix.
filtered_table_go = filter_table(table_go, features_df)


## 7. Clustered Heatmap of PASNet Pathway–Gene Weights

This cell runs a multi-step pipeline that turns the filtered weight matrix into a clustered heatmap:

1. **Fill & take absolute values** — missing values become `0`, and all weights are converted to their absolute value so the heatmap reflects connection *strength* rather than sign.
2. **Row/column filtering** — `filter_rows_cols` removes pathways with too few non-zero genes, and genes with too few non-zero pathways, controlled by `min_nonzero_rows` / `min_nonzero_cols`.
3. **Name cleanup** — `clean_source_names` strips Gene Ontology / Reactome / WikiPathways ID suffixes (e.g. `(GO:0006915)`, `R-HSA-12345`, `WP123`) from pathway names so only the human-readable label remains; one especially long pathway name is also manually shortened for display.
4. **Min–max scaling** — weights are rescaled to a common `0–1` range purely for consistent heatmap coloring.
5. **Hierarchical clustering** — `cluster_dataframe` reorders both pathways (rows) and genes (columns) by average-linkage Euclidean-distance hierarchical clustering, so visually similar weight profiles end up next to each other.
6. **Highlighting** — `color_xticklabels` bolds/colors the x-axis gene labels listed in `lncRNA_genes` (configured in Section 2).
7. **Plot** — a `seaborn` heatmap is drawn from the clustered, scaled matrix.


In [ ]:
# ===============================================================
# Fill missing gene values with 0 (numeric columns only)
# ===============================================================
numeric_cols = filtered_table_go.columns.difference(['source'])
filtered_table_go[numeric_cols] = filtered_table_go[numeric_cols].fillna(0)

# ===============================================================
# Take absolute values
# ===============================================================
# The heatmap is meant to show connection strength, so sign is dropped here.
filtered_table_go[numeric_cols] = filtered_table_go[numeric_cols].abs()

# ===============================================================
# Filter rows and columns
# ===============================================================
def filter_rows_cols(df, min_nonzero_rows=2, min_nonzero_cols=1):
    df = df.copy()
    # Keep pathways with >= min_nonzero_rows non-zero genes
    df = df[df[numeric_cols].gt(0).sum(axis=1) >= min_nonzero_rows]
    # Keep genes present in >= min_nonzero_cols pathways
    nonzero_genes = df[numeric_cols].gt(0).sum(axis=0) >= min_nonzero_cols
    df = pd.concat([df[['source']], df[numeric_cols].loc[:, nonzero_genes]], axis=1)
    return df

filtered_table_go = filter_rows_cols(filtered_table_go, min_nonzero_rows=3, min_nonzero_cols=1)

# ===============================================================
# Recompute numeric_cols after filtering
# ===============================================================
numeric_cols = filtered_table_go.columns.difference(['source'])

# ===============================================================
# Clean pathway/source names
# ===============================================================
def clean_source_names(df):
    df = df.copy()
    # Strip GO term IDs, e.g. "... (GO:0006915)"
    df['source'] = df['source'].str.replace(r"\s*\(GO:[^)]+\)", "", regex=True)
    # Strip Reactome IDs, e.g. "... R-HSA-123456"
    df['source'] = df['source'].str.replace(r"\s*R-HSA-\d+$", "", regex=True)
    # Strip WikiPathways IDs, e.g. "... WP123"
    df['source'] = df['source'].str.replace(r"\s*WP\d+$", "", regex=True)
    return df

filtered_table_go = clean_source_names(filtered_table_go)
filtered_table_go_for_next_steps = filtered_table_go

# Shorten one particularly long pathway name for display purposes.
filtered_table_go['source'] = filtered_table_go['source'].replace(
    "Diseases Of Signal Transduction By Growth Factor Receptors And Second Messengers",
    "Signal Transduction By Growth Factor Receptors, Second Messengers"
)

# ===============================================================
# Normalize numeric values (Min-Max scaling 0-1)
# ===============================================================
def scale_dataframe(df, feature_range=(0, 1)):
    data = df[numeric_cols]
    scaler = MinMaxScaler(feature_range=feature_range)
    scaled = pd.DataFrame(scaler.fit_transform(data), columns=data.columns, index=df.index)
    df_scaled = pd.concat([df[['source']], scaled], axis=1)
    return df_scaled

filtered_table_go = scale_dataframe(filtered_table_go)

# ===============================================================
# Hierarchical clustering function
# ===============================================================
def cluster_dataframe(df):
    data = df[numeric_cols]
    # Cluster rows (pathways)
    row_linkage = linkage(data, method='average', metric='euclidean')
    row_order = leaves_list(row_linkage)
    # Cluster columns (genes)
    col_linkage = linkage(data.T, method='average', metric='euclidean')
    col_order = leaves_list(col_linkage)
    # Reorder dataframe according to the clustering-derived leaf order
    clustered_df = df.iloc[row_order, :]
    clustered_df = clustered_df.iloc[:, [0] + (col_order + 1).tolist()]
    return clustered_df

clustered_filtered_table_go = cluster_dataframe(filtered_table_go)

# ===============================================================
# Highlight function for genes of interest
# ===============================================================
def color_xticklabels(ax, lncRNA_genes):
    for tick in ax.get_xticklabels():
        if tick.get_text() in lncRNA_genes:
            tick.set_color('red')
            tick.set_fontweight('bold')
        else:
            tick.set_color('black')
            tick.set_fontweight('normal')
    ax.tick_params(axis='x', rotation=90)

# ===============================================================
# Plot filtered_table_go clustered heatmap
# ===============================================================
plt.figure(figsize=(15, 12))
sns.heatmap(
    clustered_filtered_table_go.set_index('source'),
    cmap='Purples',
    annot=False,
    yticklabels=True
)
plt.title('GO PASNet Pathway-Gene Weights', fontsize=16, fontweight='bold')
plt.xlabel('Genes', fontsize=14)
plt.ylabel('Pathways (Gene Ontology)', fontsize=14)
plt.xticks(fontsize=12, rotation=90)
plt.yticks(fontsize=12)

ax = plt.gca()
color_xticklabels(ax, lncRNA_genes)

plt.tight_layout()
plt.show()


### 7.1 Pathway and Gene Lists After Clustering/Filtering


In [ ]:
# Pathway and gene identifiers retained after the filtering/clustering steps
# above; reused later to restrict the Driver/SJARACNe data to the same set.
pathways = clustered_filtered_table_go['source'].tolist()
gene_list = sorted(set(clustered_filtered_table_go.columns[1:].tolist()))

print(len(pathways))
print(len(gene_list))


## 8. Convert the Filtered Weight Matrix Back into an Edge List

For the network-graph stage we need pathway–gene relationships back in long (edge-list) form rather than the wide matrix used for the heatmap. The filtered (but unclustered, unscaled) table from Section 7 is melted accordingly, zero-weight rows are dropped, and three placeholder columns (`MI`, `spearman`, `p-value`) are added so this table has the same column structure as the SJARACNe co-expression edges loaded in Section 10 — PASNet edges simply leave those three columns empty.


In [ ]:
# Use 'source' (pathway) as the index, then immediately reset it so the melt
# below operates on a plain column rather than the index.
filtered_table_go_for_next_steps_temp = filtered_table_go_for_next_steps.set_index('source')
df_reset = filtered_table_go_for_next_steps_temp.reset_index()

# Melt the wide pathway x gene matrix into a long edge list.
long_df = df_reset.melt(id_vars='source', var_name='column_name', value_name='value')
long_df = long_df.rename(columns={'source': 'row_name'})

# Re-melt with the column names actually used downstream (source/target/weights).
long_df = df_reset.melt(id_vars='source', var_name='target', value_name='weights')

# Drop pathway-gene pairs with no PASNet weight.
long_df = long_df[long_df['weights'] != 0]

filtered_table_go_df = long_df

# Placeholder columns matching the SJARACNe edge table's schema (Section 10),
# so the two edge tables can later be concatenated.
filtered_table_go_df['MI'] = None
filtered_table_go_df['spearman'] = None
filtered_table_go_df['p-value'] = None


## 9. Load and Filter the Driver Inference Output Table

`Driver_Inference_ms_tab.xlsx` is the output of the (separate) Driver Inference pipeline step, classifying genes as transcription factors (`TF`), signaling molecules (`SIG`), or non-coding drivers (`NC`), with differential activity (`DA`) and differential expression (`DE`) statistics for each. The relevant columns are selected and renamed, duplicate gene entries are resolved (preferring the `TF` row when a gene has more than one entry), and two filtered subsets are derived:

- `res_filt_overt`: genes significant in **both** DA and DE (with `Size` above a minimum threshold).
- `res_filt_hidden`: genes significant in DA but **not** in DE (again with `Size` above the threshold).


In [ ]:
# Load the Driver Inference results and keep only the columns needed downstream.
res = pd.read_excel("../pipeline/output/driver/Driver_Inference/Driver_output/Driver_Inference/DATA/Driver_Inference_ms_tab.xlsx")
res = res[['geneSymbol', 'funcType', 'Size',
                     'logFC_DA',
                     'adj_pval_DA',
                     'logFC_DE',
                     'adj_pval_DE']]
res.rename(columns={
    'geneSymbol': 'ID',
    'funcType': 'Type',
    'logFC_DA': 'logFC_DA',
    'adj_pval_DA': 'adj.P.Val_DA',
    'logFC_DE': 'logFC_DE',
    'adj_pval_DE': 'adj.P.Val_DE',
}, inplace=True)

# A gene can appear more than once under different 'Type' classifications;
# when that happens, prefer keeping its 'TF' (transcription factor) row.
res['priority'] = res['Type'].apply(lambda x: 0 if x == 'TF' else 1)

# Sort so TF rows come first within each gene ID, then drop later duplicates.
res = res.sort_values(by=['priority', 'ID'])
res = res.drop_duplicates(subset='ID', keep='first')
res = res.drop(columns=['priority'])

# Genes significant in both differential activity (DA) and differential
# expression (DE) analyses, above a minimum cluster/group size.
res_filt_overt = res[
    (res['adj.P.Val_DA'] < 0.05) &
    (res['adj.P.Val_DE'] < 0.05) &
    (res['Size'] > 30)
]

# Genes significant in DA but not in DE, above the same size threshold.
res_filt_hidden = res[
    (res['adj.P.Val_DA'] < 0.05) &
    (res['adj.P.Val_DE'] > 0.05) &
    (res['Size'] > 30)
]

print("Overt Drivers:", len(res_filt_overt), "\n"
      "Hidden_Drivers:", len(res_filt_hidden), "\n"
      "Overt TFs:", len(res_filt_overt[res_filt_overt['Type'] == "TF"]), "\n"
      "Overt SIGs:", len(res_filt_overt[res_filt_overt['Type'] == "SIG"]), "\n"
      "Overt NCs:", len(res_filt_overt[res_filt_overt['Type'] == "NC"]), "\n"
      "Hidden TFs:", len(res_filt_hidden[res_filt_hidden['Type'] == "TF"]), "\n"
      "Hidden SIGs:", len(res_filt_hidden[res_filt_hidden['Type'] == "SIG"]), "\n"
      "Hidden NCs:", len(res_filt_hidden[res_filt_hidden['Type'] == "NC"]))


### 9.1 Restrict Driver Table to PASNet/SHAP Genes

Filters the full Driver Inference table down to genes that are **both** in the top SHAP feature list (Section 4) **and** in the clustered PASNet gene list (Section 7.1) — i.e. genes relevant to all three data sources.


In [ ]:
res_filt = res[
    res['ID'].isin(features_df['Feature']) &
    res['ID'].isin(gene_list)
]
len(res_filt)


## 10. Load and Merge SJARACNe Co-Expression Networks

[SJARACNe](https://github.com/jyyulab/SJARACNe) infers gene regulatory networks from expression data using mutual information. The pipeline produces three separate consensus networks — one each for signaling genes (`sig`), transcription factors (`tf`), and non-coding drivers (`nc`) — which are loaded and concatenated here.

Processing steps:
1. Row-bind the three sub-networks into one table.
2. Drop exact duplicate `(source, target)` edges.
3. Keep only the `source`, `target`, `MI` (mutual information), `spearman`, and `p-value` columns.
4. Add an empty `Weight` column so the schema matches the PASNet edge table.
5. Keep only statistically significant edges (`p-value <= 0.05`).
6. Since the underlying network is undirected, `(A, B)` and `(B, A)` can both appear as separate rows; these are collapsed by sorting each pair alphabetically and keeping only the highest-MI duplicate.
7. Concatenate the resulting SJARACNe edges with the PASNet pathway–gene edges from Section 8 into one combined edge table.


In [ ]:
# Load the three SJARACNe consensus co-expression networks.
df_sj_sig = pd.read_csv("../pipeline/output/sjaracne/SJARACNe/output_sig_sjaracne_Driver_Inference_out_.final/consensus_network_ncol_.txt", sep='\t')
df_sj_tf = pd.read_csv("../pipeline/output/sjaracne/SJARACNe/output_tf_sjaracne_Driver_Inference_out_.final/consensus_network_ncol_.txt", sep='\t')
df_sj_nc = pd.read_csv("../pipeline/output/sjaracne/SJARACNe/output_nc_sjaracne_Driver_Inference_out_.final/consensus_network_ncol_.txt", sep='\t')

# Row-bind the three sub-networks into a single table.
df_sj = pd.concat([df_sj_sig, df_sj_tf, df_sj_nc], axis=0, ignore_index=True)

# Remove exact duplicate (source, target) edges.
df_sj = df_sj.drop_duplicates(subset=['source', 'target'])

# Keep only the columns needed for the combined network.
columns_to_keep = ['source', 'target', 'MI', 'spearman', 'p-value']
df_sj = df_sj[columns_to_keep]

# Harmonize schema with the PASNet edge table (which has a 'Weight' column
# but no MI/spearman/p-value of its own).
df_sj['Weight'] = None
df_sj = df_sj[df_sj['p-value'] <= 0.05]

# Collapse undirected duplicate pairs (A,B) / (B,A) by sorting each pair and
# keeping only the highest-MI occurrence of each unique pair.
df_sj['sorted_pair'] = df_sj.apply(lambda row: tuple(sorted([row['source'], row['target']])), axis=1)
df_sj = df_sj.sort_values(by='MI', ascending=False).drop_duplicates(subset=['sorted_pair']).drop(columns=['sorted_pair'])

# Combine SJARACNe co-expression edges with the PASNet pathway-gene edges.
filtered_table_go_df_sj = pd.concat([filtered_table_go_df, df_sj], ignore_index=True)


## 11. Build the Node-Attribute Table

Before constructing the graph, every node referenced in the combined edge table needs an attribute row (its `Type`, and where available its DA/DE statistics). Two sources are combined:

- **Pathway nodes**: built directly from the unique `source` values in the combined edge table that correspond to pathways, with `Type` set to `"Pathway"` and all numeric statistics columns left as `NaN` (pathways don't have DA/DE statistics).
- **Gene/TF/SIG/NC nodes**: the filtered Driver Inference table (`res_filt`, Section 9.1), which already has `Type` and DA/DE columns populated.

These two tables are concatenated into one combined node-attribute table.


In [ ]:
# 1. Build a node-attribute table for pathway nodes.
weights = filtered_table_go_df_sj['source'].unique()
weights_df = pd.DataFrame({
    'ID': weights,
    'Type': "Pathway",
    'Size': np.nan,
    'logFC_DA': np.nan,
    'adj.P.Val_DA': np.nan,
    'Z-score_DA': np.nan,
    'logFC_DE': np.nan,
    'adj.P.Val_DE': np.nan,
    'Z-score_DE': np.nan
})

# 2. Keep only pathway-node entries that match the pathway names retained
#    after the heatmap clustering/filtering step (Section 7.1).
weights_df_filtered = weights_df[
    weights_df['ID'].apply(lambda x: any(go in x for go in pathways))
]

# 3. Concatenate pathway-node attributes with gene/TF/SIG/NC node attributes.
node_filtered_table_go = pd.concat([res_filt, weights_df_filtered], ignore_index=True)
len(node_filtered_table_go)


## 12. Network Visualization

The remaining cells build a `networkx` graph from the combined edge and node tables, then filter, lay out, and draw it as a community-clustered network plot.


### 12.1 Construct the Graph Object

1. Restrict the edge table to edges whose `source` and `target` both have a matching entry in the node-attribute table (so every edge endpoint can be attributed).
2. Build an **undirected** `networkx.Graph`, adding each edge with its full row of attributes (`Weight`, `MI`, `spearman`, `p-value`).
3. Attach each node's attributes (`Type`, DA/DE statistics, etc.) from the node table.
4. Add a `label` attribute to every node equal to its name, used for plot labeling later.


In [ ]:
# Step 1: Edge table.
df_edges = filtered_table_go_df_sj

# Step 2: Node attribute table.
df_node_attributes = node_filtered_table_go

# Keep only edges whose endpoints both have a matching node-attribute entry.
df_edges = df_edges[df_edges['source'].isin(df_node_attributes['ID']) & df_edges['target'].isin(df_node_attributes['ID'])]

# Step 3: Create an undirected graph.
G = nx.Graph()

# Step 4: Add edges, carrying over their attribute columns (Weight/MI/etc.).
for index, row in df_edges.iterrows():
    source = row['source']
    target = row['target']
    attributes = row.drop(['source', 'target'])  # Extract edge attributes
    G.add_edge(source, target, **attributes)  # Add edge with attributes

# Step 5: Transfer node attributes onto the matching graph nodes.
for index, row in df_node_attributes.iterrows():
    node_name = row['ID']
    if node_name in G.nodes:
        attributes = row.drop('ID')  # Remove the node name to keep only the attributes
        G.nodes[node_name].update(attributes)

# Add a 'label' attribute (= node name) for use in the plot below.
for node in G.nodes:
    G.nodes[node]['label'] = str(node)

print(f"Edges: {len(G.edges)}")
print(f"Nodes: {len(G.nodes)}")


### 12.2 Filter, Cluster, Lay Out, and Draw the Network

This cell takes the full graph `G` and renders the final figure:

1. **`clean_label`** strips pathway database IDs (Reactome `R-HSA-...`, WikiPathways `WP...`, GO `(GO:...)`) from node names for display.
2. **Pathway filtering** keeps only pathway nodes with at least `min_degree_pathway` connections, plus all `TF` / `SIG` / `NC` nodes, and builds the induced subgraph `F_filtered` on that node set.
3. **Centrality** computes betweenness centrality on `F_filtered` (used both to size nodes and to rank pathways).
4. **Top pathways** selects the `top_n_pathways` pathway nodes with highest betweenness centrality, to be labeled in the plot (labeling every pathway would be unreadable).
5. **Label cleanup** builds a node -> display-label mapping for all `TF` / `SIG` / `NC` nodes and the selected top pathways.
6. **Community detection** partitions `F_filtered` into communities using the Louvain algorithm (`python-louvain`), falling back to NetworkX's greedy modularity communities if that package isn't available.
7. **Layout** arranges each community around its own centroid on a circle, then runs a local spring layout within each community, so communities are visually separated while internal structure is preserved.
8. **Node styling** colors nodes by community (with `NC` nodes always colored yellow-orange regardless of community), and sizes nodes by their betweenness centrality.
9. **Drawing** renders nodes, curved edges, and the selected labels (with `adjustText` used to reduce label overlap), plus a legend mapping colors to communities/`NC`.


In [ ]:

F = G  # operate on a reference to the graph built in the previous cell
label_types = {'NC', 'TF', 'SIG'}
top_n_pathways = 25
min_degree_pathway = 2

# ---------------------
#  FUNCTIONS
# ---------------------
def clean_label(label):
    """Remove pathway database IDs (Reactome, WikiPathways, GO) from a label."""
    patterns = [r'R-HSA-\d+', r'WP\d+', r'\(GO:\d+\)']
    for pattern in patterns:
        label = re.sub(pattern, '', label)
    return label.strip()

# ---------------------
#  FILTER PATHWAYS
# ---------------------
# Keep only pathway nodes with enough connections to be visually meaningful.
pathways_to_keep = [
    n for n in F.nodes()
    if F.nodes[n]['Type'] == 'Pathway' and F.degree(n) >= min_degree_pathway
]

# Keep all TF/SIG/NC nodes, plus the selected pathway nodes.
nodes_to_keep = [
    n for n in F.nodes()
    if F.nodes[n]['Type'] in label_types or n in pathways_to_keep
]

F_filtered = F.subgraph(nodes_to_keep).copy()

# ---------------------
#  CENTRALITY
# ---------------------
# Approximate betweenness centrality (k=10 sampled source nodes for speed).
centrality = nx.betweenness_centrality(F_filtered, k=10, endpoints=True)

pathway_centrality = {
    n: centrality[n]
    for n in F_filtered.nodes()
    if F_filtered.nodes[n]['Type'] == 'Pathway'
}

# Pathways selected to be labeled in the final plot.
top_pathways = sorted(pathway_centrality, key=pathway_centrality.get, reverse=True)[:top_n_pathways]

# ---------------------
#  CLEAN LABELS
# ---------------------
cleaned_labels = {}
for node in F_filtered.nodes():
    if F_filtered.nodes[node]['Type'] in label_types or node in top_pathways:
        cleaned_labels[node] = clean_label(node)

# ---------------------
#  COMMUNITY DETECTION
# ---------------------
# Prefer the Louvain algorithm; fall back to greedy modularity communities
# if the 'community' (python-louvain) package isn't installed.
try:
    import community as community_louvain
    partition = community_louvain.best_partition(F_filtered)
except:
    communities_list = list(nx.algorithms.community.greedy_modularity_communities(F_filtered))
    partition = {}
    for i, comm in enumerate(communities_list):
        for node in comm:
            partition[node] = i

# ---------------------
#  COMMUNITY CLUSTERED LAYOUT
# ---------------------
# Group nodes by their assigned community.
communities = {}
for node, comm_id in partition.items():
    communities.setdefault(comm_id, []).append(node)

# Place each community's centroid evenly around a circle...
radius = 3
num_comms = len(communities)
angle_step = 2 * np.pi / num_comms
comm_centroids = {
    comm_id: (radius * np.cos(i * angle_step), radius * np.sin(i * angle_step))
    for i, comm_id in enumerate(communities)
}

# ...then run a local spring layout within each community and offset it to
# sit around that community's centroid, so communities are visually separated.
pos = {}
for comm_id, nodes in communities.items():
    subgraph = F_filtered.subgraph(nodes)
    sub_pos = nx.spring_layout(subgraph, k=0.8, seed=1234)
    cx, cy = comm_centroids[comm_id]
    for n, (x, y) in sub_pos.items():
        pos[n] = (x + cx, y + cy)

# ---------------------
#  NODE COLORS & SIZES
# ---------------------
# Map each community to a distinct color, but always color NC nodes
# yellow-orange regardless of which community they fall into.
comm_ids = sorted(set(partition.values()))
cmap = plt.get_cmap('tab20', len(comm_ids))

node_color = []
for n in F_filtered.nodes():
    if F_filtered.nodes[n]['Type'] == 'NC':
        node_color.append('#FFBE00')  # NC nodes highlighted
    else:
        node_color.append(cmap(partition[n]))

# Node size scales with betweenness centrality.
node_size = [centrality[v] * 5000 + 100 for v in F_filtered.nodes()]

# ---------------------
#  DRAW GRAPH
# ---------------------
fig, ax = plt.subplots(figsize=(14, 7))

# Draw nodes
nx.draw_networkx_nodes(
    F_filtered,
    pos=pos,
    node_color=node_color,
    node_size=node_size,
    alpha=0.6
)

# Draw curved edges (slightly arced rather than straight lines, for readability).
for u, v in F_filtered.edges():
    ax.annotate(
        '',
        xy=pos[v], xycoords='data',
        xytext=pos[u], textcoords='data',
        arrowprops=dict(
            arrowstyle='-',
            color='gainsboro',
            lw=0.7,
            connectionstyle='arc3,rad=0.2'
        )
    )

# ---------------------
#  ADD LABELS
# ---------------------
texts = []
for node, label in cleaned_labels.items():
    x, y = pos[node]
    text = ax.text(
        x, y, label,
        fontsize=10, color='black',
        ha='center', va='center',
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, pad=1)
    )
    texts.append(text)

# Nudge overlapping labels apart and draw thin leader lines back to their nodes.
adjust_text(
    texts,
    arrowprops=dict(arrowstyle='-', color='black', lw=0.5),
    expand_points=(1.3, 1.3),
    force_points=0.5,
    force_text=0.5
)

# ---------------------
#  LEGEND
# ---------------------
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#FFBE00', markersize=10)]
labels = ['NC']

# Add one representative legend entry per community (excluding the NC color).
for i in comm_ids:
    if not any(F_filtered.nodes[n]['Type']=='NC' for n in communities[i]):
        handles.append(plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=cmap(i), markersize=10))
        labels.append(f"Community {i}")

#ax.legend(handles=handles, labels=labels, loc='upper left', fontsize=12)

# ---------------------
#  TITLE & SAVE
# ---------------------
plt.title("Bipartite Graph lncRNA-mRNA-Pathway Correlations", fontsize=15, fontweight='bold')
ax.margins(0.1, 0.05)
fig.tight_layout()
plt.axis("off")
plt.show()


## 13. List Node Membership Per Detected Community

Using the `partition` mapping computed during the network plot, this cell groups nodes by community ID and prints the cleaned, de-duplicated label list for each one.


In [ ]:
# ---------------------
# UNIQUE LIST OF NODES PER CLUSTER
# ---------------------
clusters = {}
for node, comm_id in partition.items():
    clusters.setdefault(comm_id, []).append(node)

print("\n=== UNIQUE NODE LISTS PER CLUSTER ===\n")

for comm_id, nodes in sorted(clusters.items()):
    # Convert to cleaned labels (falling back to the raw node name if unlabeled).
    unique_clean_labels = sorted({cleaned_labels.get(n, n) for n in nodes})

    print(f"Cluster {comm_id}:")
    print(unique_clean_labels)
    print()
